## 1. Read CSV File into Data Frame and Display Schema

In [8]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Read_CSV').getOrCreate()
df=spark.read.option('header', True).option('inferSchema', True).csv("data/employee.csv")

df.printSchema()


root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: timestamp (nullable = true)



## 2. Filter Rows where a column value is greater than a threshold

In [9]:
#df.show()

threshold=60000
df_filtered=df.filter(df["salary"]>threshold)
df_filtered.show(5, truncate=False)

+------+------------------+------------+-------+-------------------+
|emp_id|emp_name          |department  |salary |join_date          |
+------+------------------+------------+-------+-------------------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|
|2     |Aaron M. Tenenbaum| Engineering|75000.0|2021-06-01 00:00:00|
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|
+------+------------------+------------+-------+-------------------+



## 3 select specific columns from DataFrame

In [10]:
df_selected=df.select('emp_id', 'emp_name', 'salary')
df_selected.show(5, truncate=False)

+------+------------------+-------+
|emp_id|emp_name          |salary |
+------+------------------+-------+
|1     |A. Kannan         |70000.0|
|2     |Aaron M. Tenenbaum|75000.0|
|3     |Adam Smith        |76000.0|
|4     |Adity Gupta       |60000.0|
|5     |Alex lonescu      |50000.0|
+------+------------------+-------+
only showing top 5 rows


## 4 Rename One or more Columns

In [11]:
# rename single column
df_renamed=df.withColumnRenamed('emp_name', 'employee_name') \
    .withColumnRenamed('salary','annual_salary')
df_renamed.show(5, truncate=False)

+------+------------------+------------+-------------+-------------------+
|emp_id|employee_name     |department  |annual_salary|join_date          |
+------+------------------+------------+-------------+-------------------+
|1     |A. Kannan         | Engineering|70000.0      |2021-06-15 00:00:00|
|2     |Aaron M. Tenenbaum| Engineering|75000.0      |2021-06-01 00:00:00|
|3     |Adam Smith        | Marketing  |76000.0      |2021-07-19 00:00:00|
|4     |Adity Gupta       | Engineering|60000.0      |2021-03-17 00:00:00|
|5     |Alex lonescu      | Sales      |50000.0      |2021-02-18 00:00:00|
+------+------------------+------------+-------------+-------------------+
only showing top 5 rows


## 5 Add a new derived column using withColumn

In [12]:
# Add a new column: Bonus= 10% of salary

from pyspark.sql.functions import col
df_with_bonus=df.withColumn('Bonus',(col('salary')*0.10).cast('double'))
df_with_bonus.show(10, truncate=False)

+------+------------------+------------+-------+-------------------+------+
|emp_id|emp_name          |department  |salary |join_date          |Bonus |
+------+------------------+------------+-------+-------------------+------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|7000.0|
|2     |Aaron M. Tenenbaum| Engineering|75000.0|2021-06-01 00:00:00|7500.0|
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|7600.0|
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|6000.0|
|5     |Alex lonescu      | Sales      |50000.0|2021-02-18 00:00:00|5000.0|
|6     |Alex Mackman      | Engineering|40000.0|2021-01-19 00:00:00|4000.0|
|7     |Alfred.V.Aho      | Sales      |45000.0|2021-11-25 00:00:00|4500.0|
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|3500.0|
|9     |Ananth Grama      | Marketing  |30000.0|2021-03-20 00:00:00|3000.0|
|10    |Andre Tost        | Sales      |25000.0|2021-05-11 00:00:00|2500.0|
+------+----

## 6 Drop Duplicate Rows

In [13]:
df_dedup=df.dropDuplicates()
df_dedup.show(5,truncate=False)

+------+------------------+------------+-------+-------------------+
|emp_id|emp_name          |department  |salary |join_date          |
+------+------------------+------------+-------+-------------------+
|8     |Alon Fliess       | Engineering|35000.0|2021-12-05 00:00:00|
|7     |Alfred.V.Aho      | Sales      |45000.0|2021-11-25 00:00:00|
|2     |Aaron M. Tenenbaum| Engineering|75000.0|2021-06-01 00:00:00|
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|
+------+------------------+------------+-------+-------------------+
only showing top 5 rows


## 7 Count null values in each column

In [14]:
from pyspark.sql.functions import col,sum as _sum, when,lit

null_count_df=df.select([_sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in df.columns])
null_count_df.show(truncate=False)

+------+--------+----------+------+---------+
|emp_id|emp_name|department|salary|join_date|
+------+--------+----------+------+---------+
|0     |0       |0         |0     |0        |
+------+--------+----------+------+---------+



## 8 Replace Null with default values

In [15]:
from pyspark.sql.functions import lit
df_filled=df.fillna({
    'emp_name':'Unknown',
    'department': 'Not Assigned',
    'Salary':0.0,
    'join_date':'1970-01-01'
})
df_filled.show(5, truncate=False)


+------+------------------+------------+-------+-------------------+
|emp_id|emp_name          |department  |salary |join_date          |
+------+------------------+------------+-------+-------------------+
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|
|2     |Aaron M. Tenenbaum| Engineering|75000.0|2021-06-01 00:00:00|
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|
|4     |Adity Gupta       | Engineering|60000.0|2021-03-17 00:00:00|
|5     |Alex lonescu      | Sales      |50000.0|2021-02-18 00:00:00|
+------+------------------+------------+-------+-------------------+
only showing top 5 rows


## 9 Use groupBy and agg to compute sum, count, average, min, max

In [16]:
from pyspark.sql.functions import sum as _sum,count, avg, min as _min, max as _max
agg_df=df.groupBy('department').agg(
    _sum('salary').alias('total_Salary'),
    count('*').alias('count'),
    avg('salary').alias('avg_salary'),
    _min('salary').alias('min_salary'),
    _max('salary').alias('max_salary')
).orderBy('department')

agg_df.show(truncate=False)

+------------+------------+-----+----------+----------+----------+
|department  |total_Salary|count|avg_salary|min_salary|max_salary|
+------------+------------+-----+----------+----------+----------+
| Engineering|280000.0    |5    |56000.0   |35000.0   |75000.0   |
| Marketing  |106000.0    |2    |53000.0   |30000.0   |76000.0   |
| Sales      |120000.0    |3    |40000.0   |25000.0   |50000.0   |
+------------+------------+-----+----------+----------+----------+



## 10 Find the top N records by a metric

In [17]:
N=3
topn_df=df.orderBy(col('salary').desc()) \
    .limit(N)

topn_df.show(truncate=False)

+------+------------------+------------+-------+-------------------+
|emp_id|emp_name          |department  |salary |join_date          |
+------+------------------+------------+-------+-------------------+
|3     |Adam Smith        | Marketing  |76000.0|2021-07-19 00:00:00|
|2     |Aaron M. Tenenbaum| Engineering|75000.0|2021-06-01 00:00:00|
|1     |A. Kannan         | Engineering|70000.0|2021-06-15 00:00:00|
+------+------------------+------------+-------+-------------------+

